In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -U wandb

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch wandb API key from Kaggle secrets
secrets = UserSecretsClient()
wandb_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset, Audio
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import os, random
import numpy as np
import librosa
from tqdm import tqdm

In [ ]:
BASE_DIR   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR  = os.path.join(BASE_DIR, 'genres_stems')
ESC50_DIR  = os.path.join(BASE_DIR, 'ESC-50-master/audio')
SAVE_DIR   = '/kaggle/working'

# Feature Extraction


In [ ]:
SAMPLE_RATE  = 22050
DURATION     = 30
N_MELS       = 128
N_FFT        = 2048
HOP_LENGTH   = 512
TARGET_SHAPE = (128, 512)

def wav_to_mel(audio):
    mel    = librosa.feature.melspectrogram(
        y          = audio,
        sr         = SAMPLE_RATE,
        n_mels     = N_MELS,
        n_fft      = N_FFT,
        hop_length = HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=1.0)
    return mel_db

In [ ]:

GENRES     = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2IDX  = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE  = {i: g for g, i in GENRE2IDX.items()}
NUM_CLASSES = 10

SEED = 42
random.seed(SEED)
np.random.seed(SEED)





def fix_shape(mel):
    n_mels, time_frames = TARGET_SHAPE
    if mel.shape[1] >= time_frames:
        mel = mel[:, :time_frames]
    else:
        mel = np.pad(mel, ((0, 0), (0, time_frames - mel.shape[1])), mode='constant')
    return mel


# FUNCTION 1 — Multicrop from file
# loads wav, converts to mel, extracts 3 crops (start, center, end)
# returns 3 arrays each of shape (128, 512)


def multicrop_from_file(file_path):
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
    target_len = SAMPLE_RATE * DURATION

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    total  = len(audio)
    crop_len = int(total * 0.8)

    starts = [
        0,
        (total - crop_len) // 2,
        total - crop_len
    ]

    crops = []
    for start in starts:
        crop = audio[start : start + crop_len]
        mel  = wav_to_mel(crop)
        mel  = fix_shape(mel)
        crops.append(mel)

    return crops


# FUNCTION 2 — Multicrop from waveform
# same as above but takes waveform array directly
# used for synthetic mashups already in memory
def multicrop_from_waveform(audio):
    target_len = SAMPLE_RATE * DURATION

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    total    = len(audio)
    crop_len = int(total * 0.8)

    starts = [
        0,
        (total - crop_len) // 2,
        total - crop_len
    ]

    crops = []
    for start in starts:
        crop = audio[start : start + crop_len]
        mel  = wav_to_mel(crop)
        mel  = fix_shape(mel)
        crops.append(mel)

    return crops


# FUNCTION 3 — Synthetic mashup creator
# picks 4 random songs from same genre
# takes one stem from each (drums, vocals, bass, other)
# applies random tempo augmentation +-5% with 70% probability
# sums waveforms together and normalizes
def create_synthetic_mashup(genre):
    genre_path   = os.path.join(STEMS_DIR, genre)
    song_folders = sorted(os.listdir(genre_path))
    stem_names   = ['drums', 'vocals', 'bass', 'other']
    target_len   = SAMPLE_RATE * DURATION
    mixed        = None

    selected_songs = random.sample(song_folders, min(4, len(song_folders)))

    for song_name, stem_name in zip(selected_songs, stem_names):
        song_path = os.path.join(genre_path, song_name)
        stem_path = os.path.join(song_path, stem_name + '.wav')

        if not os.path.exists(stem_path):
            wav_files = [f for f in os.listdir(song_path) if f.endswith('.wav')]
            if len(wav_files) == 0:
                continue
            stem_path = os.path.join(song_path, random.choice(wav_files))

        try:
            audio, _ = librosa.load(stem_path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
        except Exception:
            continue

        # Apply tempo augmentation +-5% with 70% probability
        if random.random() < 0.7:
            rate  = random.uniform(0.95, 1.05)
            audio = librosa.effects.time_stretch(audio, rate=rate)

        if len(audio) < target_len:
            audio = np.pad(audio, (0, target_len - len(audio)))
        else:
            audio = audio[:target_len]

        if mixed is None:
            mixed = audio.copy()
        else:
            mixed = mixed + audio

    if mixed is None:
        return None

    # Normalize to prevent clipping
    max_val = np.max(np.abs(mixed))
    if max_val > 0:
        mixed = mixed / max_val

    return mixed


# FUNCTION 4 — Noise injector
# picks 1 to 4 random ESC-50 clips
# inserts each at random position with random intensity 0.05 to 0.3
# normalizes after injection
def inject_noise(audio):
    target_len  = SAMPLE_RATE * DURATION
    noise_files = [f for f in os.listdir(ESC50_DIR) if f.endswith('.wav')]
    num_noises  = random.randint(1, 4)

    for _ in range(num_noises):
        noise_file = random.choice(noise_files)
        noise_path = os.path.join(ESC50_DIR, noise_file)

        try:
            noise, _ = librosa.load(noise_path, sr=SAMPLE_RATE, mono=True)
        except Exception:
            continue

        if len(noise) < target_len:
            noise = np.pad(noise, (0, target_len - len(noise)))
        else:
            noise = noise[:target_len]

        # Insert at random position
        insert_pos = random.randint(0, target_len - len(noise[:target_len]))
        noise_clip = np.zeros(target_len)
        clip_len   = min(len(noise), target_len - insert_pos)
        noise_clip[insert_pos : insert_pos + clip_len] = noise[:clip_len]

        intensity = random.uniform(0.05, 0.3)
        audio     = audio + intensity * noise_clip

    # Normalize after injection
    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio / max_val

    return audio


print("All 4 feature extraction functions defined!")
print("multicrop_from_file     — ready")
print("multicrop_from_waveform — ready")
print("create_synthetic_mashup — ready")
print("inject_noise            — ready")

All 4 feature extraction functions defined!

multicrop_from_file     — ready

multicrop_from_waveform — ready

create_synthetic_mashup — ready

inject_noise            — ready

# Injecting noises and creating messy dataset

In [ ]:
"""# BLOCK 2 — BUILD DATASET
# Part A: 100 songs x 4 stems x 3 crops = 1200 real samples per genre
# Part B: 500 synthetic mashups x 3 crops = 1500 synthetic samples per genre
# Total: 2700 per genre x 10 genres = 27000 samples

X_list = []
y_list = []

for genre in GENRES:
    print()
    print("Processing genre:", genre)
    genre_path   = os.path.join(STEMS_DIR, genre)
    song_folders = sorted(os.listdir(genre_path))
    genre_idx    = GENRE2IDX[genre]

    # ── PART A — Real stems ───────────────────────────────────
    print("  Part A: processing", len(song_folders), "songs x 4 stems x 3 crops...")
    part_a_count = 0

    for song_name in tqdm(song_folders, desc="  " + genre + " real stems"):
        song_path  = os.path.join(genre_path, song_name)
        stem_names = ['drums', 'vocals', 'bass', 'other']

        for stem_name in stem_names:
            stem_path = os.path.join(song_path, stem_name + '.wav')

            if not os.path.exists(stem_path):
                wav_files = [f for f in os.listdir(song_path) if f.endswith('.wav')]
                if len(wav_files) == 0:
                    continue
                stem_path = os.path.join(song_path, random.choice(wav_files))

            try:
                crops = multicrop_from_file(stem_path)
                for crop in crops:
                    X_list.append(crop)
                    y_list.append(genre_idx)
                part_a_count += 3
            except Exception as e:
                continue

    print("  Part A done:", part_a_count, "samples added")

    # ── PART B — Synthetic mashups ────────────────────────────
    print("  Part B: generating 500 synthetic mashups x 3 crops...")
    part_b_count = 0

    for _ in tqdm(range(500), desc="  " + genre + " synthetic"):
        try:
            audio = create_synthetic_mashup(genre)
            if audio is None:
                continue

            audio = inject_noise(audio)
            crops = multicrop_from_waveform(audio)

            for crop in crops:
                X_list.append(crop)
                y_list.append(genre_idx)
            part_b_count += 3

        except Exception as e:
            continue

    print("  Part B done:", part_b_count, "samples added")
    print("  Genre total:", part_a_count + part_b_count, "samples")

print()
print("All genres processed!")
print("Total samples in list:", len(X_list))

# Fix shape inconsistencies — some crops may be off due to tempo augmentation
print()
print("Fixing shape inconsistencies...")
X_list = [
    x[:, :512] if x.shape[1] >= 512
    else np.pad(x, ((0, 0), (0, 512 - x.shape[1])), mode='constant')
    for x in X_list
]

print("Converting to numpy arrays...")
X_train_final = np.array(X_list, dtype=np.float32)
y_train_final = np.array(y_list, dtype=np.int64)

del X_list, y_list

print()
print("X_train_final shape:", X_train_final.shape)
print("y_train_final shape:", y_train_final.shape)
print("Label distribution:", {IDX2GENRE[i]: int(np.sum(y_train_final == i)) for i in range(NUM_CLASSES)})

print()
print("Saving to disk...")
np.save(os.path.join(SAVE_DIR, 'X_train_final.npy'), X_train_final)
np.save(os.path.join(SAVE_DIR, 'y_train_final.npy'), y_train_final)
print("Saved X_train_final.npy and y_train_final.npy!")
print("DONE!")"""

Processing genre: blues

  Part A: processing 100 songs x 4 stems x 3 crops...

  
  blues real stems: 100%|██████████| 100/100 [01:31<00:00,  1.09it/s]
  
  Part A done: 1200 samples added
  
  Part B: generating 500 synthetic mashups x 3 crops...
  
  blues synthetic: 100%|██████████| 500/500 [06:41<00:00,  1.25it/s]
  
  Part B done: 1500 samples added
  
  Genre total: 2700 samples
  

Processing genre: classical

  Part A: processing 100 songs x 4 stems x 3 crops...
  
  classical real stems: 100%|██████████| 100/100 [01:18<00:00,  1.28it/s]
  
  Part A done: 1200 samples added

  
  Part B: generating 500 synthetic mashups x 3 crops...
  
  classical synthetic: 100%|██████████| 500/500 [06:45<00:00,  1.23it/s]
  
  Part B done: 1500 samples added
  Genre total: 2700 samples

Processing genre: country
  Part A: processing 100 songs x 4 stems x 3 crops...
  country real stems: 100%|██████████| 100/100 [01:21<00:00,  1.23it/s]
  Part A done: 1200 samples added
  Part B: generating 500 synthetic mashups x 3 crops...
  country synthetic: 100%|██████████| 500/500 [06:40<00:00,  1.25it/s]
  Part B done: 1500 samples added
  Genre total: 2700 samples

Processing genre: disco


  Part A: processing 100 songs x 4 stems x 3 crops...
  
  disco real stems: 100%|██████████| 100/100 [01:20<00:00,  1.24it/s]
  
  Part A done: 1200 samples added
  
  Part B: generating 500 synthetic mashups x 3 crops...
  
  disco synthetic: 100%|██████████| 500/500 [06:31<00:00,  1.28it/s]
  
  Part B done: 1500 samples added
  
  Genre total: 2700 samples
  

Processing genre: hiphop

  Part A: processing 100 songs x 4 stems x 3 crops...
  
  hiphop real stems: 100%|██████████| 100/100 [01:33<00:00,  1.07it/s]
  
  Part A done: 1200 samples added
  
  Part B: generating 500 synthetic mashups x 3 crops...
  
  hiphop synthetic: 100%|██████████| 500/500 [06:40<00:00,  1.25it/s]
  
  Part B done: 1500 samples added
  
  Genre total: 2700 samples
  

Processing genre: jazz

  Part A: processing 100 songs x 4 stems x 3 crops...
  
  jazz real stems: 100%|██████████| 100/100 [01:25<00:00,  1.17it/s]
  
  Part A done: 1200 samples added
  
  Part B: generating 500 synthetic mashups x 3 crops...
  
  jazz synthetic: 100%|██████████| 500/500 [06:33<00:00,  1.27it/s]
  
  Part B done: 1500 samples added
  
  Genre total: 2700 samples
  

Processing genre: metal

  Part A: processing 100 songs x 4 stems x 3 crops...
  
  metal real stems: 100%|██████████| 100/100 [01:21<00:00,  1.23it/s]
  
  Part A done: 1200 samples added
  
  Part B: generating 500 synthetic mashups x 3 crops...
  
  metal synthetic: 100%|██████████| 500/500 [06:55<00:00,  1.20it/s]
  
  Part B done: 1500 samples added
  
  Genre total: 2700 samples
  

Processing genre: pop

  Part A: processing 100 songs x 4 stems x 3 crops...
  
  pop real stems: 100%|██████████| 100/100 [01:25<00:00,  1.18it/s]

  
  Part A done: 1200 samples added
  
   Part B: generating 500 synthetic mashups x 3 crops...
   
  pop synthetic: 100%|██████████| 500/500 [06:32<00:00,  1.27it/s]
  
  Part B done: 1500 samples added
  
  Genre total: 2700 samples


Processing genre: reggae
  Part A: processing 100 songs x 4 stems x 3 crops...
  
  reggae real stems: 100%|██████████| 100/100 [01:22<00:00,  1.21it/s]
  
  Part A done: 1200 samples added
  
  Part B: generating 500 synthetic mashups x 3 crops...
  
  reggae synthetic: 100%|██████████| 500/500 [06:25<00:00,  1.30it/s]
  
  Part B done: 1500 samples added
  
  Genre total: 2700 samples
  

Processing genre: rock

  Part A: processing 100 songs x 4 stems x 3 crops...
  
  rock real stems: 100%|██████████| 100/100 [01:28<00:00,  1.13it/s]
  Part A done: 1200 samples added
  
  Part B: generating 500 synthetic mashups x 3 crops...
  
  rock synthetic: 100%|██████████| 500/500 [06:51<00:00,  1.22it/s]
  
  Part B done: 1500 samples added
  
  Genre total: 2700 samples


All genres processed!

Total samples in list: 27000


Fixing shape inconsistencies...

Converting to numpy arrays...


X_train_final shape: (27000, 128, 512)

y_train_final shape: (27000,)

Label distribution: {'blues': 2700, 'classical': 2700, 'country': 2700, 'disco': 2700, 'hiphop': 2700, 'jazz': 2700, 'metal': 2700, 'pop': 2700, 'reggae': 2700, 'rock': 2700}


Saving to disk...

Saved X_train_final.npy and y_train_final.npy!
DONE!

# CNN


# CNN Dataloader

In [ ]:
"""# ============================================================
# SCRATCH CNN TRAINING BLOCK — SELF CONTAINED
# ============================================================

import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm
import wandb
from kaggle_secrets import UserSecretsClient

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

GENRES      = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2IDX   = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE   = {i: g for g, i in GENRE2IDX.items()}
NUM_CLASSES = 10
BATCH_SIZE  = 32
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR    = '/kaggle/working'



# ── Load Data ────────────────────────────────────────────────
DATASET_PATH  = '/kaggle/input/datasets/shalini784683raaih/dataset-mashed'
X_train_final = np.load(os.path.join(DATASET_PATH, 'X_train_final-DL.npy'), mmap_mode='r')
y_train_final = np.load(os.path.join(DATASET_PATH, 'y_train_final.npy'))

print("X_train_final shape:", X_train_final.shape)
print("y_train_final shape:", y_train_final.shape)

indices            = np.arange(len(X_train_final))
train_idx, val_idx = train_test_split(
    indices, test_size=0.15, stratify=y_train_final, random_state=SEED
)

mean = float(np.mean(X_train_final))
std  = float(np.std(X_train_final))
print("Mean:", round(mean, 4), "| Std:", round(std, 4))
print("Train samples:", len(train_idx), "| Val samples:", len(val_idx))

# ── Dataset ──────────────────────────────────────────────────
class AudioDataset(Dataset):
    def __init__(self, X, y, mean, std, augment=False):
        self.X       = X
        self.y       = y
        self.mean    = mean
        self.std     = std
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        mel = torch.tensor(self.X[idx], dtype=torch.float32)
        mel = (mel - self.mean) / (self.std + 1e-6)
        if self.augment:
            if random.random() < 0.5:
                f_start = random.randint(0, 108)
                f_end   = random.randint(f_start + 1, min(f_start + 20, 128))
                mel[f_start:f_end, :] = 0
            if random.random() < 0.5:
                t_start = random.randint(0, 472)
                t_end   = random.randint(t_start + 1, min(t_start + 40, 512))
                mel[:, t_start:t_end] = 0
            if random.random() < 0.5:
                mel = mel * random.uniform(0.8, 1.2)
        mel = mel.unsqueeze(0)
        return mel, torch.tensor(self.y[idx], dtype=torch.long)


train_dataset = AudioDataset(X_train_final[train_idx], y_train_final[train_idx], mean, std, augment=True)
val_dataset   = AudioDataset(X_train_final[val_idx],   y_train_final[val_idx],   mean, std, augment=False)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

sample_batch, _ = next(iter(train_loader))
print("Batch shape:", sample_batch.shape)
print("Expected   : torch.Size([32, 1, 128, 512])")

# ── Model ────────────────────────────────────────────────────
class GenreCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(GenreCNN, self).__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        x = self.classifier(x)
        return x


cnn_model = GenreCNN(num_classes=NUM_CLASSES).to(DEVICE)

total     = sum(p.numel() for p in cnn_model.parameters())
trainable = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print("Total params    :", f"{total:,}")
print("Trainable params:", f"{trainable:,}")

with torch.no_grad():
    out = cnn_model(sample_batch.to(DEVICE))
print("Output shape:", out.shape)

# ── Training ─────────────────────────────────────────────────
EPOCHS    = 50
LR        = 1e-3
PATIENCE  = 10
best_f1   = 0.0
no_improve= 0
best_path = os.path.join(SAVE_DIR, 'best_cnn.pth')

wandb.init(
    project = "DL_GENAI_AUDIO",
    name    = "scratch-cnn-v1",
    config  = {
        "model"        : "ScratchCNN",
        "epochs"       : EPOCHS,
        "lr"           : LR,
        "batch_size"   : BATCH_SIZE,
        "patience"     : PATIENCE,
        "train_samples": len(train_idx),
        "val_samples"  : len(val_idx),
        "augmentation" : True,
    },
    reinit = True
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(cnn_model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

print()
print("Starting Scratch CNN training...")
print("Epochs:", EPOCHS, "| LR:", LR, "| Patience:", PATIENCE)
print()

for epoch in range(1, EPOCHS + 1):

    cnn_model.train()
    train_loss, train_preds, train_targets = 0.0, [], []

    for bx, by in tqdm(train_loader, desc="Epoch " + str(epoch) + " Train", leave=False):
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out  = cnn_model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
        train_targets.extend(by.cpu().numpy())

    avg_train_loss = train_loss / len(train_loader)
    train_f1       = f1_score(train_targets, train_preds, average='macro')

    cnn_model.eval()
    val_loss, val_preds, val_targets = 0.0, [], []

    with torch.no_grad():
        for bx, by in tqdm(val_loader, desc="Epoch " + str(epoch) + " Val", leave=False):
            bx, by   = bx.to(DEVICE), by.to(DEVICE)
            out       = cnn_model(bx)
            loss      = criterion(out, by)
            val_loss += loss.item()
            val_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            val_targets.extend(by.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_f1       = f1_score(val_targets, val_preds, average='macro')

    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1    = val_f1
        no_improve = 0
        torch.save(cnn_model.state_dict(), best_path)
        print("  New best saved! Val F1:", round(val_f1, 4))
    else:
        no_improve += 1
        print("  No improvement for", no_improve, "epochs")

    wandb.log({
        "epoch"      : epoch,
        "train_loss" : avg_train_loss,
        "val_loss"   : avg_val_loss,
        "train_f1"   : train_f1,
        "val_f1"     : val_f1,
        "lr"         : optimizer.param_groups[0]['lr']
    })

    print("Epoch", epoch, "/", EPOCHS,
          "| Train Loss:", round(avg_train_loss, 4),
          "| Val Loss:", round(avg_val_loss, 4),
          "| Train F1:", round(train_f1, 4),
          "| Val F1:", round(val_f1, 4))

    if no_improve >= PATIENCE:
        print("Early stopping triggered!")
        break

print()
print("Training Complete! Best Val F1:", round(best_f1, 4))
print("Model saved to:", best_path)
wandb.finish()"""

X_train_final shape: (27000, 128, 512)
y_train_final shape: (27000,)
Mean: -52.4154 | Std: 19.3747
Train samples: 22950 | Val samples: 4050
Batch shape: torch.Size([32, 1, 128, 512])
Expected   : torch.Size([32, 1, 128, 512])
Total params    : 2,491,594
Trainable params: 2,491,594
Output shape: torch.Size([32, 10])
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
Tracking run with wandb version 0.25.1
Run data is saved locally in /kaggle/working/wandb/run-20260401_130432-53ak1vva
Syncing run scratch-cnn-v1 to Weights & Biases (docs)
View project at https://wandb.ai/23f2005169-indian-institute-of-technology-madras/DL_GENAI_Audio
View run at https://wandb.ai/23f2005169-indian-institute-of-technology-madras/DL_GENAI_Audio/runs/53ak1vva

Starting Scratch CNN training...
Epochs: 50 | LR: 0.001 | Patience: 10

                                                                
  New best saved! Val F1: 0.3745
Epoch 1 / 50 | Train Loss: 2.0897 | Val Loss: 1.7936 | Train F1: 0.2355 | Val F1: 0.3745
                                                                
  New best saved! Val F1: 0.4831
Epoch 2 / 50 | Train Loss: 1.912 | Val Loss: 1.653 | Train F1: 0.3295 | Val F1: 0.4831
                                                                
  New best saved! Val F1: 0.5126
Epoch 3 / 50 | Train Loss: 1.8433 | Val Loss: 1.6136 | Train F1: 0.3666 | Val F1: 0.5126
                                                                
  New best saved! Val F1: 0.5736
Epoch 4 / 50 | Train Loss: 1.7784 | Val Loss: 1.5018 | Train F1: 0.4051 | Val F1: 0.5736
                                                                
  No improvement for 1 epochs
Epoch 5 / 50 | Train Loss: 1.7306 | Val Loss: 1.4625 | Train F1: 0.4339 | Val F1: 0.5499
                                                                
  New best saved! Val F1: 0.6214
Epoch 6 / 50 | Train Loss: 1.6876 | Val Loss: 1.3978 | Train F1: 0.4621 | Val F1: 0.6214
                                                                
  No improvement for 1 epochs
Epoch 7 / 50 | Train Loss: 1.645 | Val Loss: 1.3601 | Train F1: 0.4872 | Val F1: 0.5951
                                                                
  New best saved! Val F1: 0.6451
Epoch 8 / 50 | Train Loss: 1.5935 | Val Loss: 1.3099 | Train F1: 0.5171 | Val F1: 0.6451
                                                                
  New best saved! Val F1: 0.6581
Epoch 9 / 50 | Train Loss: 1.5542 | Val Loss: 1.3097 | Train F1: 0.5379 | Val F1: 0.6581
                                                                 
  No improvement for 1 epochs
Epoch 10 / 50 | Train Loss: 1.5167 | Val Loss: 1.2854 | Train F1: 0.5575 | Val F1: 0.6576
                                                                 
  New best saved! Val F1: 0.7171
Epoch 11 / 50 | Train Loss: 1.4813 | Val Loss: 1.2089 | Train F1: 0.5739 | Val F1: 0.7171
                                                                 
  New best saved! Val F1: 0.7223
Epoch 12 / 50 | Train Loss: 1.4556 | Val Loss: 1.2021 | Train F1: 0.5847 | Val F1: 0.7223
                                                                 
  New best saved! Val F1: 0.7324
Epoch 13 / 50 | Train Loss: 1.4406 | Val Loss: 1.1767 | Train F1: 0.5975 | Val F1: 0.7324
                                                                 
  New best saved! Val F1: 0.7487
Epoch 14 / 50 | Train Loss: 1.4092 | Val Loss: 1.1343 | Train F1: 0.6128 | Val F1: 0.7487
                                                                 
  No improvement for 1 epochs
Epoch 15 / 50 | Train Loss: 1.389 | Val Loss: 1.1253 | Train F1: 0.6215 | Val F1: 0.7462
                                                                 
  New best saved! Val F1: 0.7614
Epoch 16 / 50 | Train Loss: 1.3798 | Val Loss: 1.1131 | Train F1: 0.6298 | Val F1: 0.7614
                                                                 
  No improvement for 1 epochs
Epoch 17 / 50 | Train Loss: 1.3606 | Val Loss: 1.1211 | Train F1: 0.6384 | Val F1: 0.7566
                                                                 
  New best saved! Val F1: 0.7658
Epoch 18 / 50 | Train Loss: 1.347 | Val Loss: 1.0952 | Train F1: 0.647 | Val F1: 0.7658
                                                                 
  New best saved! Val F1: 0.7797
Epoch 19 / 50 | Train Loss: 1.333 | Val Loss: 1.07 | Train F1: 0.6552 | Val F1: 0.7797
                                                                 
  No improvement for 1 epochs
Epoch 20 / 50 | Train Loss: 1.329 | Val Loss: 1.0792 | Train F1: 0.656 | Val F1: 0.7703

                                                                 
  New best saved! Val F1: 0.7928
Epoch 21 / 50 | Train Loss: 1.3119 | Val Loss: 1.0515 | Train F1: 0.6644 | Val F1: 0.7928
                                                                 
  No improvement for 1 epochs
Epoch 22 / 50 | Train Loss: 1.2995 | Val Loss: 1.0621 | Train F1: 0.6693 | Val F1: 0.7728
                                                                 
  New best saved! Val F1: 0.7979
Epoch 23 / 50 | Train Loss: 1.2927 | Val Loss: 1.0268 | Train F1: 0.6733 | Val F1: 0.7979
                                                                 
  New best saved! Val F1: 0.7993
Epoch 24 / 50 | Train Loss: 1.2839 | Val Loss: 1.0402 | Train F1: 0.6788 | Val F1: 0.7993
                                                                 
  No improvement for 1 epochs
Epoch 25 / 50 | Train Loss: 1.2759 | Val Loss: 1.0469 | Train F1: 0.6808 | Val F1: 0.7861
                                                                 
  New best saved! Val F1: 0.8089
Epoch 26 / 50 | Train Loss: 1.2697 | Val Loss: 1.003 | Train F1: 0.688 | Val F1: 0.8089
                                                                 
  No improvement for 1 epochs
Epoch 27 / 50 | Train Loss: 1.2603 | Val Loss: 1.0136 | Train F1: 0.687 | Val F1: 0.8019
                                                                 
  No improvement for 2 epochs
Epoch 28 / 50 | Train Loss: 1.2552 | Val Loss: 1.0273 | Train F1: 0.6924 | Val F1: 0.8026
                                                                 
  New best saved! Val F1: 0.8223
Epoch 29 / 50 | Train Loss: 1.2506 | Val Loss: 0.9884 | Train F1: 0.6953 | Val F1: 0.8223
                                                                 
  No improvement for 1 epochs
Epoch 30 / 50 | Train Loss: 1.2393 | Val Loss: 1.0136 | Train F1: 0.6992 | Val F1: 0.7947
                                                                 
  No improvement for 2 epochs
Epoch 31 / 50 | Train Loss: 1.2453 | Val Loss: 1.0106 | Train F1: 0.7008 | Val F1: 0.8122
                                                                 
  No improvement for 3 epochs
Epoch 32 / 50 | Train Loss: 1.2377 | Val Loss: 0.9876 | Train F1: 0.703 | Val F1: 0.8185
                                                                 
  No improvement for 4 epochs
Epoch 33 / 50 | Train Loss: 1.2313 | Val Loss: 1.0045 | Train F1: 0.7054 | Val F1: 0.8041
                                                                 
  No improvement for 5 epochs
Epoch 34 / 50 | Train Loss: 1.2306 | Val Loss: 0.9947 | Train F1: 0.706 | Val F1: 0.8064
                                                                 
  New best saved! Val F1: 0.8464
Epoch 35 / 50 | Train Loss: 1.155 | Val Loss: 0.9232 | Train F1: 0.7412 | Val F1: 0.8464
                                                                 
  No improvement for 1 epochs
Epoch 36 / 50 | Train Loss: 1.1381 | Val Loss: 0.9228 | Train F1: 0.7512 | Val F1: 0.8447
                                                                 
  New best saved! Val F1: 0.8562
Epoch 37 / 50 | Train Loss: 1.1293 | Val Loss: 0.9089 | Train F1: 0.756 | Val F1: 0.8562
                                                                 
  No improvement for 1 epochs
Epoch 38 / 50 | Train Loss: 1.125 | Val Loss: 0.9265 | Train F1: 0.7579 | Val F1: 0.8466
                                                                 
  New best saved! Val F1: 0.8568
Epoch 39 / 50 | Train Loss: 1.1184 | Val Loss: 0.8968 | Train F1: 0.7612 | Val F1: 0.8568
                                                                 
  No improvement for 1 epochs
Epoch 40 / 50 | Train Loss: 1.1118 | Val Loss: 0.9166 | Train F1: 0.765 | Val F1: 0.8491
                                                                 
  New best saved! Val F1: 0.862
Epoch 41 / 50 | Train Loss: 1.1067 | Val Loss: 0.893 | Train F1: 0.7665 | Val F1: 0.862
                                                                 
  No improvement for 1 epochs
Epoch 42 / 50 | Train Loss: 1.1038 | Val Loss: 0.9038 | Train F1: 0.7671 | Val F1: 0.8559
                                                                 
  New best saved! Val F1: 0.8653
Epoch 43 / 50 | Train Loss: 1.1026 | Val Loss: 0.8934 | Train F1: 0.7718 | Val F1: 0.8653
                                                                 
  No improvement for 1 epochs
Epoch 44 / 50 | Train Loss: 1.0906 | Val Loss: 0.9019 | Train F1: 0.774 | Val F1: 0.8534

<more outputs >




## Trainig loop 

## Inference 

In [ ]:
"""# ============================================================
# SCRATCH CNN INFERENCE — FULLY SELF CONTAINED
# comment out entire notebook and run only this block
# ============================================================

import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from tqdm import tqdm

GENRES       = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2IDX    = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE    = {i: g for g, i in GENRE2IDX.items()}
NUM_CLASSES  = 10
SAMPLE_RATE  = 22050
DURATION     = 30
N_MELS       = 128
N_FFT        = 2048
HOP_LENGTH   = 512
TARGET_SHAPE = (128, 512)
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR     = '/kaggle/working'

BASE_DIR   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
TEST_CSV   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv'
MODEL_PATH = '/kaggle/working/best_cnn.pth'

MEAN = -52.4154
STD  =  19.3747

print("Device:", DEVICE)
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


def wav_to_mel(audio):
    mel    = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db


def fix_shape(mel):
    n_mels, time_frames = TARGET_SHAPE
    if mel.shape[1] >= time_frames:
        mel = mel[:, :time_frames]
    else:
        mel = np.pad(mel, ((0, 0), (0, time_frames - mel.shape[1])), mode='constant')
    return mel


class GenreCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(GenreCNN, self).__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        x = self.classifier(x)
        return x


def multicrop_inference(model, file_path):
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
    target_len = SAMPLE_RATE * DURATION

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    total    = len(audio)
    crop_len = int(total * 0.8)
    starts   = [0, (total - crop_len) // 2, total - crop_len]

    vote_scores = np.zeros(NUM_CLASSES)
    for start in starts:
        crop = audio[start : start + crop_len]
        mel  = wav_to_mel(crop)
        mel  = fix_shape(mel)
        mel  = torch.tensor(mel, dtype=torch.float32)
        mel  = (mel - MEAN) / (STD + 1e-6)
        mel  = mel.unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            vote_scores += torch.softmax(model(mel), dim=1).cpu().numpy()[0]
    return vote_scores


cnn_model = GenreCNN(num_classes=NUM_CLASSES)
cnn_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
cnn_model = cnn_model.to(DEVICE)
cnn_model.eval()
print("CNN loaded from:", MODEL_PATH)

test_df = pd.read_csv(TEST_CSV)
print("Total test files:", len(test_df))

predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    file_path = os.path.join(BASE_DIR, row['filename'])

    if not os.path.exists(file_path):
        print("Missing:", file_path)
        predictions.append({'id': row['id'], 'genre': 'pop'})
        continue

    try:
        scores    = multicrop_inference(cnn_model, file_path)
        predicted = IDX2GENRE[np.argmax(scores)]
        predictions.append({'id': row['id'], 'genre': predicted})
    except Exception as e:
        print("Error on", row['filename'], ":", e)
        predictions.append({'id': row['id'], 'genre': 'pop'})

submission_df = pd.DataFrame(predictions).sort_values('id').reset_index(drop=True)
submission_df.to_csv(os.path.join(SAVE_DIR, 'submission_cnn.csv'), index=False)

print()
print("Genre distribution:")
print(submission_df['genre'].value_counts())
print()
print("Saved submission_cnn.csv!")
print("DONE!")"""

Device: cuda
GPU   : Tesla T4
CNN loaded from: /kaggle/working/best_cnn.pth
Total test files: 3020
Inference: 100%|██████████| 3020/3020 [05:58<00:00,  8.43it/s]

Genre distribution:
genre
disco        633

blues        418

country      340

rock         305

metal        303

hiphop       272

pop          248

reggae       188

jazz         171

classical    142

Name: count, dtype: int64

Saved submission_cnn.csv!
DONE!


# Resnet 50

## Resnet dataloader 

In [ ]:
"""#class audio dataset code + loading pretrained resnet 50 from pytorch
# RESNET DATALOADER — AudioDataset + ResNet50 model

import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torchvision import models
from tqdm import tqdm
import wandb
from kaggle_secrets import UserSecretsClient

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

GENRES      = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2IDX   = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE   = {i: g for g, i in GENRE2IDX.items()}
NUM_CLASSES = 10
BATCH_SIZE  = 32
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR    = '/kaggle/working'

print("Device:", DEVICE)
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

secrets   = UserSecretsClient()
wandb_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)
print("WandB ready!")

class AudioDataset(Dataset):
    def __init__(self, X, y, mean, std, augment=False):
        self.X       = X
        self.y       = y
        self.mean    = mean
        self.std     = std
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        mel = torch.tensor(self.X[idx], dtype=torch.float32)
        mel = (mel - self.mean) / (self.std + 1e-6)

        if self.augment:
            if random.random() < 0.5:
                f_start = random.randint(0, 108)
                f_end   = random.randint(f_start + 1, min(f_start + 20, 128))
                mel[f_start:f_end, :] = 0
            if random.random() < 0.5:
                t_start = random.randint(0, 472)
                t_end   = random.randint(t_start + 1, min(t_start + 40, 512))
                mel[:, t_start:t_end] = 0

        mel = mel.unsqueeze(0).repeat(3, 1, 1)
        return mel, torch.tensor(self.y[idx], dtype=torch.long)


print()
print("Loading data from /kaggle/working...")
X_train_final = np.load(os.path.join(SAVE_DIR, 'X_train_final.npy'), mmap_mode='r')
y_train_final = np.load(os.path.join(SAVE_DIR, 'y_train_final.npy'))

print("X shape:", X_train_final.shape)
print("y shape:", y_train_final.shape)

mean = float(np.mean(X_train_final))
std  = float(np.std(X_train_final))
print("Mean:", round(mean, 4))
print("Std :", round(std,  4))

indices            = np.arange(len(X_train_final))
train_idx, val_idx = train_test_split(
    indices,
    test_size    = 0.15,
    stratify     = y_train_final,
    random_state = SEED
)

print("Train samples:", len(train_idx))
print("Val samples  :", len(val_idx))

train_dataset = AudioDataset(X_train_final[train_idx], y_train_final[train_idx], mean, std, augment=True)
val_dataset   = AudioDataset(X_train_final[val_idx],   y_train_final[val_idx],   mean, std, augment=False)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

sample_batch, sample_labels = next(iter(train_loader))
print()
print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Batch X shape:", sample_batch.shape)
print("Expected     : torch.Size([32, 3, 128, 512])")

resnet_model       = models.resnet50(weights='IMAGENET1K_V1')
resnet_model.fc    = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(2048, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, NUM_CLASSES)
)

for param in resnet_model.parameters():
    param.requires_grad = False
for param in resnet_model.layer4.parameters():
    param.requires_grad = True
for param in resnet_model.fc.parameters():
    param.requires_grad = True

resnet_model = resnet_model.to(DEVICE)

total     = sum(p.numel() for p in resnet_model.parameters())
trainable = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print()
print("Total params    :", f"{total:,}")
print("Trainable params:", f"{trainable:,}")
print("DONE!")"""

## Resnet Training loop

In [ ]:
"""resnet_model.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet_model = resnet_model.to(DEVICE)"""

In [ ]:
"""#training loop with wandb initializations with all optimizers and schedulers
# RESNET TRAINING LOOP

EPOCHS    = 25
LR        = 3e-4
PATIENCE  = 8
best_f1   = 0.0
no_improve= 0
best_path = os.path.join(SAVE_DIR, 'best_resnet50.pth')

wandb.init(
    project = "DL_GENAI_AUDIO",
    name    = "resnet50-v1",
    config  = {
        "model"        : "ResNet50",
        "epochs"       : EPOCHS,
        "lr"           : LR,
        "batch_size"   : BATCH_SIZE,
        "patience"     : PATIENCE,
        "train_samples": len(train_idx),
        "val_samples"  : len(val_idx),
        "augmentation" : True,
        "frozen"       : "layer1,layer2,layer3",
        "trainable"    : "layer4,fc"
    },
    reinit = True
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model.parameters()),
    lr=LR, weight_decay=1e-3
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=4, min_lr=1e-6
)

print("Starting ResNet50 training...")
print("Epochs:", EPOCHS, "| LR:", LR, "| Patience:", PATIENCE)
print()

for epoch in range(1, EPOCHS + 1):

    resnet_model.train()
    train_loss, train_preds, train_targets = 0.0, [], []

    for bx, by in tqdm(train_loader, desc="Epoch " + str(epoch) + " Train", leave=False):
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out  = resnet_model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
        train_targets.extend(by.cpu().numpy())

    avg_train_loss = train_loss / len(train_loader)
    train_f1       = f1_score(train_targets, train_preds, average='macro')

    resnet_model.eval()
    val_loss, val_preds, val_targets = 0.0, [], []

    with torch.no_grad():
        for bx, by in tqdm(val_loader, desc="Epoch " + str(epoch) + " Val", leave=False):
            bx, by  = bx.to(DEVICE), by.to(DEVICE)
            out      = resnet_model(bx)
            loss     = criterion(out, by)
            val_loss += loss.item()
            val_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            val_targets.extend(by.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_f1       = f1_score(val_targets, val_preds, average='macro')

    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1    = val_f1
        no_improve = 0
        torch.save(resnet_model.state_dict(), best_path)
        print("  New best saved! Val F1:", round(val_f1, 4))
    else:
        no_improve += 1
        print("  No improvement for", no_improve, "epochs")

    wandb.log({
        "epoch"      : epoch,
        "train_loss" : avg_train_loss,
        "val_loss"   : avg_val_loss,
        "train_f1"   : train_f1,
        "val_f1"     : val_f1,
        "lr"         : optimizer.param_groups[0]['lr']
    })

    print("Epoch", epoch, "/", EPOCHS,
          "| Train Loss:", round(avg_train_loss, 4),
          "| Val Loss:", round(avg_val_loss, 4),
          "| Train F1:", round(train_f1, 4),
          "| Val F1:", round(val_f1, 4))

    if no_improve >= PATIENCE:
        print("Early stopping triggered!")
        break

print()
print("Training Complete!")
print("Best Val F1:", round(best_f1, 4))
print("Model saved to:", best_path)
wandb.finish()"""

## Resnet inference and submission

In [ ]:
"""import numpy as np

DATASET_PATH  = '/kaggle/working'
X_train_final = np.load(os.path.join(DATASET_PATH, 'X_train_final.npy'), mmap_mode='r')

mean = float(np.mean(X_train_final))
std  = float(np.std(X_train_final))

print("Correct MEAN:", round(mean, 4))
print("Correct STD :", round(std, 4))
print("Update these in your inference block!")"""

In [ ]:
"""# RESNET INFERENCE AND SUBMISSION — FULLY SELF CONTAINED

import os, random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm

GENRES       = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2IDX    = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE    = {i: g for g, i in GENRE2IDX.items()}
NUM_CLASSES  = 10
SAMPLE_RATE  = 22050
DURATION     = 30
N_MELS       = 128
N_FFT        = 2048
HOP_LENGTH   = 512
TARGET_SHAPE = (128, 512)
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR     = '/kaggle/working'

BASE_DIR   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
TEST_CSV   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv'
SUB_CSV    = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv'
MODEL_PATH = '/kaggle/working/best_resnet50.pth'

MEAN = -52.4154
STD  =  19.3747

print("Device:", DEVICE)
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


def wav_to_mel(audio):
    mel    = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db


def fix_shape(mel):
    n_mels, time_frames = TARGET_SHAPE
    if mel.shape[1] >= time_frames:
        mel = mel[:, :time_frames]
    else:
        mel = np.pad(mel, ((0, 0), (0, time_frames - mel.shape[1])), mode='constant')
    return mel


def multicrop_inference(model, file_path):
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
    target_len = SAMPLE_RATE * DURATION

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    total    = len(audio)
    crop_len = int(total * 0.8)
    starts   = [0, (total - crop_len) // 2, total - crop_len]

    vote_scores = np.zeros(NUM_CLASSES)
    for start in starts:
        crop = audio[start : start + crop_len]
        mel  = wav_to_mel(crop)
        mel  = fix_shape(mel)
        mel  = torch.tensor(mel, dtype=torch.float32)
        mel  = (mel - MEAN) / (STD + 1e-6)
        mel  = mel.unsqueeze(0).repeat(3, 1, 1)
        mel  = mel.unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            vote_scores += torch.softmax(model(mel), dim=1).cpu().numpy()[0]
    return vote_scores


resnet_model       = models.resnet50(weights=None)
resnet_model.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet_model.fc    = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(2048, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, NUM_CLASSES)
)

resnet_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
resnet_model = resnet_model.to(DEVICE)
resnet_model.eval()
print("Model loaded from:", MODEL_PATH)

# Read test.csv which has id and filename columns
test_df = pd.read_csv(TEST_CSV)
sub_df  = pd.read_csv(SUB_CSV)
print("Total test files:", len(test_df))
print("Test CSV columns:", test_df.columns.tolist())

predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    # filename column already has relative path like mashups/song0001.wav
    file_path = os.path.join(BASE_DIR, row['filename'])

    if not os.path.exists(file_path):
        print("Missing:", file_path)
        predictions.append({'id': row['id'], 'genre': 'pop'})
        continue

    try:
        scores    = multicrop_inference(resnet_model, file_path)
        predicted = IDX2GENRE[np.argmax(scores)]
        predictions.append({'id': row['id'], 'genre': predicted})
    except Exception as e:
        print("Error on", row['filename'], ":", e)
        predictions.append({'id': row['id'], 'genre': 'pop'})

submission_df = pd.DataFrame(predictions).sort_values('id').reset_index(drop=True)
submission_df.to_csv(os.path.join(SAVE_DIR, 'submission.csv'), index=False)

print()
print("Genre distribution:")
print(submission_df['genre'].value_counts())
print()
print("Saved submission_resnet.csv!")
print("DONE!")"""

# Efficient Net 

In [ ]:
"""# ============================================================
# EFFICIENTNET-B3 TRAINING BLOCK — SELF CONTAINED
# ============================================================

import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm
import timm
import wandb
from kaggle_secrets import UserSecretsClient

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

GENRES      = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2IDX   = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE   = {i: g for g, i in GENRE2IDX.items()}
NUM_CLASSES = 10
BATCH_SIZE  = 32
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR    = '/kaggle/working'

print("Device:", DEVICE)
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

secrets   = UserSecretsClient()
wandb_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

# ── Load Data ────────────────────────────────────────────────
DATASET_PATH  = '/kaggle/input/datasets/shalini784683raaih/dataset-mashed'
X_train_final = np.load(os.path.join(DATASET_PATH, 'X_train_final_DL.npy'), mmap_mode='r')
y_train_final = np.load(os.path.join(DATASET_PATH, 'y_train_final.npy'))

print("X_train_final shape:", X_train_final.shape)
print("y_train_final shape:", y_train_final.shape)

indices            = np.arange(len(X_train_final))
train_idx, val_idx = train_test_split(
    indices, test_size=0.15, stratify=y_train_final, random_state=SEED
)

mean = float(np.mean(X_train_final))
std  = float(np.std(X_train_final))
print("Mean:", round(mean, 4), "| Std:", round(std, 4))
print("Train samples:", len(train_idx), "| Val samples:", len(val_idx))

# ── Dataset ──────────────────────────────────────────────────
class AudioDataset(Dataset):
    def __init__(self, X, y, mean, std, augment=False):
        self.X       = X
        self.y       = y
        self.mean    = mean
        self.std     = std
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        mel = torch.tensor(self.X[idx], dtype=torch.float32)
        mel = (mel - self.mean) / (self.std + 1e-6)
        if self.augment:
            if random.random() < 0.5:
                f_start = random.randint(0, 108)
                f_end   = random.randint(f_start + 1, min(f_start + 20, 128))
                mel[f_start:f_end, :] = 0
            if random.random() < 0.5:
                t_start = random.randint(0, 472)
                t_end   = random.randint(t_start + 1, min(t_start + 40, 512))
                mel[:, t_start:t_end] = 0
            if random.random() < 0.5:
                mel = mel * random.uniform(0.8, 1.2)
        mel = mel.unsqueeze(0).repeat(3, 1, 1)
        return mel, torch.tensor(self.y[idx], dtype=torch.long)


train_dataset = AudioDataset(X_train_final[train_idx], y_train_final[train_idx], mean, std, augment=True)
val_dataset   = AudioDataset(X_train_final[val_idx],   y_train_final[val_idx],   mean, std, augment=False)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

sample_batch, _ = next(iter(train_loader))
print("Batch shape:", sample_batch.shape)

# ── Model ────────────────────────────────────────────────────
eff_model = timm.create_model(
    'efficientnet_b3-2',
    pretrained  = True,
    num_classes = NUM_CLASSES,
    in_chans    = 3
)

for param in eff_model.parameters():
    param.requires_grad = False

for param in eff_model.blocks[-3:].parameters():
    param.requires_grad = True
for param in eff_model.conv_head.parameters():
    param.requires_grad = True
for param in eff_model.bn2.parameters():
    param.requires_grad = True
for param in eff_model.classifier.parameters():
    param.requires_grad = True

eff_model = eff_model.to(DEVICE)

total     = sum(p.numel() for p in eff_model.parameters())
trainable = sum(p.numel() for p in eff_model.parameters() if p.requires_grad)
print("Total params    :", f"{total:,}")
print("Trainable params:", f"{trainable:,}")

with torch.no_grad():
    out = eff_model(sample_batch.to(DEVICE))
print("Output shape:", out.shape)

# ── Training ─────────────────────────────────────────────────
EPOCHS    = 15
LR        = 3e-5
PATIENCE  = 7
best_f1   = 0.0
no_improve= 0
best_path = os.path.join(SAVE_DIR, 'best_efficientnet_b3-2.pth')

wandb.init(
    project = "DL_GENAI_AUDIO",
    name    = "efficientnet-b3-v1",
    config  = {
        "model"        : "EfficientNet-B3",
        "epochs"       : EPOCHS,
        "lr"           : LR,
        "batch_size"   : BATCH_SIZE,
        "patience"     : PATIENCE,
        "train_samples": len(train_idx),
        "val_samples"  : len(val_idx),
        "augmentation" : True,
    },
    reinit = True
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, eff_model.parameters()),
    lr=LR, weight_decay=1e-2
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

print()
print("Starting EfficientNet-B3 training...")
print("Epochs:", EPOCHS, "| LR:", LR, "| Patience:", PATIENCE)
print()

for epoch in range(1, EPOCHS + 1):

    eff_model.train()
    train_loss, train_preds, train_targets = 0.0, [], []

    for bx, by in tqdm(train_loader, desc="Epoch " + str(epoch) + " Train", leave=False):
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out  = eff_model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
        train_targets.extend(by.cpu().numpy())

    avg_train_loss = train_loss / len(train_loader)
    train_f1       = f1_score(train_targets, train_preds, average='macro')

    eff_model.eval()
    val_loss, val_preds, val_targets = 0.0, [], []

    with torch.no_grad():
        for bx, by in tqdm(val_loader, desc="Epoch " + str(epoch) + " Val", leave=False):
            bx, by   = bx.to(DEVICE), by.to(DEVICE)
            out       = eff_model(bx)
            loss      = criterion(out, by)
            val_loss += loss.item()
            val_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            val_targets.extend(by.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_f1       = f1_score(val_targets, val_preds, average='macro')

    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1    = val_f1
        no_improve = 0
        torch.save(eff_model.state_dict(), best_path)
        print("  New best saved! Val F1:", round(val_f1, 4))
    else:
        no_improve += 1
        print("  No improvement for", no_improve, "epochs")

    wandb.log({
        "epoch"      : epoch,
        "train_loss" : avg_train_loss,
        "val_loss"   : avg_val_loss,
        "train_f1"   : train_f1,
        "val_f1"     : val_f1,
        "lr"         : optimizer.param_groups[0]['lr']
    })
    print("Epoch", epoch, "/", EPOCHS,
          "| Train Loss:", round(avg_train_loss, 4),
          "| Val Loss:", round(avg_val_loss, 4),
          "| Train F1:", round(train_f1, 4),
          "| Val F1:", round(val_f1, 4))

    if no_improve >= PATIENCE:
        print("Early stopping triggered!")
        break

print()
print("Training Complete! Best Val F1:", round(best_f1, 4))
print("Model saved to:", best_path)
wandb.finish()"""

In [ ]:
"""# ============================================================
# EFFICIENTNET-B3 CONTINUE TRAINING FROM CHECKPOINT
# ============================================================

import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm
import timm
import wandb
from kaggle_secrets import UserSecretsClient

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

GENRES      = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2IDX   = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE   = {i: g for g, i in GENRE2IDX.items()}
NUM_CLASSES = 10
BATCH_SIZE  = 32
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR    = '/kaggle/working'

print("Device:", DEVICE)
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

secrets   = UserSecretsClient()
wandb_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

# ── Load Data ────────────────────────────────────────────────
DATASET_PATH  = '/kaggle/working/'
CHECKPOINT    = '/kaggle/input/datasets/shalini784683raaih/models-to-use/best_efficientnet_b3.pth'

X_train_final = np.load(os.path.join(DATASET_PATH, 'X_train_final.npy'), mmap_mode='r')
y_train_final = np.load(os.path.join(DATASET_PATH, 'y_train_final.npy'))

print("X_train_final shape:", X_train_final.shape)

indices            = np.arange(len(X_train_final))
train_idx, val_idx = train_test_split(
    indices, test_size=0.15, stratify=y_train_final, random_state=SEED
)

mean = float(np.mean(X_train_final))
std  = float(np.std(X_train_final))
print("Mean:", round(mean, 4), "| Std:", round(std, 4))
print("Train samples:", len(train_idx), "| Val samples:", len(val_idx))

# ── Dataset ──────────────────────────────────────────────────
class AudioDataset(Dataset):
    def __init__(self, X, y, mean, std, augment=False):
        self.X       = X
        self.y       = y
        self.mean    = mean
        self.std     = std
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        mel = torch.tensor(self.X[idx], dtype=torch.float32)
        mel = (mel - self.mean) / (self.std + 1e-6)
        if self.augment:
            if random.random() < 0.5:
                f_start = random.randint(0, 108)
                f_end   = random.randint(f_start + 1, min(f_start + 20, 128))
                mel[f_start:f_end, :] = 0
            if random.random() < 0.5:
                t_start = random.randint(0, 472)
                t_end   = random.randint(t_start + 1, min(t_start + 40, 512))
                mel[:, t_start:t_end] = 0
            if random.random() < 0.5:
                mel = mel * random.uniform(0.8, 1.2)
        mel = mel.unsqueeze(0).repeat(3, 1, 1)
        return mel, torch.tensor(self.y[idx], dtype=torch.long)


train_dataset = AudioDataset(X_train_final[train_idx], y_train_final[train_idx], mean, std, augment=True)
val_dataset   = AudioDataset(X_train_final[val_idx],   y_train_final[val_idx],   mean, std, augment=False)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

sample_batch, _ = next(iter(train_loader))
print("Batch shape:", sample_batch.shape)

# ── Load Model from Checkpoint ───────────────────────────────
eff_model = timm.create_model(
    'efficientnet_b3',
    pretrained  = False,
    num_classes = NUM_CLASSES,
    in_chans    = 3
)

eff_model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
eff_model = eff_model.to(DEVICE)
print("Loaded checkpoint from:", CHECKPOINT)

# Unfreeze last 3 blocks + head for continued training
for param in eff_model.parameters():
    param.requires_grad = False
for param in eff_model.blocks[-3:].parameters():
    param.requires_grad = True
for param in eff_model.conv_head.parameters():
    param.requires_grad = True
for param in eff_model.bn2.parameters():
    param.requires_grad = True
for param in eff_model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in eff_model.parameters() if p.requires_grad)
print("Trainable params:", f"{trainable:,}")

# ── Continue Training at lower LR ────────────────────────────
EPOCHS    = 20
LR        = 1e-5
PATIENCE  = 5
best_f1   = 0.0
no_improve= 0
best_path = os.path.join(SAVE_DIR, 'best_efficientnet_b3_v2.pth')

wandb.init(
    project = "DL_GENAI_AUDIO",
    name    = "efficientnet-b3-continued",
    config  = {
        "model"        : "EfficientNet-B3-continued",
        "epochs"       : EPOCHS,
        "lr"           : LR,
        "batch_size"   : BATCH_SIZE,
        "patience"     : PATIENCE,
        "checkpoint"   : CHECKPOINT,
    },
    reinit = True
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, eff_model.parameters()),
    lr=LR, weight_decay=1e-3
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

print()
print("Continuing EfficientNet-B3 training from checkpoint...")
print("Epochs:", EPOCHS, "| LR:", LR, "| Patience:", PATIENCE)
print()

for epoch in range(1, EPOCHS + 1):

    eff_model.train()
    train_loss, train_preds, train_targets = 0.0, [], []

    for bx, by in tqdm(train_loader, desc="Epoch " + str(epoch) + " Train", leave=False):
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out  = eff_model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
        train_targets.extend(by.cpu().numpy())

    avg_train_loss = train_loss / len(train_loader)
    train_f1       = f1_score(train_targets, train_preds, average='macro')

    eff_model.eval()
    val_loss, val_preds, val_targets = 0.0, [], []

    with torch.no_grad():
        for bx, by in tqdm(val_loader, desc="Epoch " + str(epoch) + " Val", leave=False):
            bx, by   = bx.to(DEVICE), by.to(DEVICE)
            out       = eff_model(bx)
            loss      = criterion(out, by)
            val_loss += loss.item()
            val_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            val_targets.extend(by.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_f1       = f1_score(val_targets, val_preds, average='macro')

    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1    = val_f1
        no_improve = 0
        torch.save(eff_model.state_dict(), best_path)
        print("  New best saved! Val F1:", round(val_f1, 4))
    else:
        no_improve += 1
        print("  No improvement for", no_improve, "epochs")

    wandb.log({
        "epoch"      : epoch,
        "train_loss" : avg_train_loss,
        "val_loss"   : avg_val_loss,
        "train_f1"   : train_f1,
        "val_f1"     : val_f1,
        "lr"         : optimizer.param_groups[0]['lr']
    })

    print("Epoch", epoch, "/", EPOCHS,
          "| Train Loss:", round(avg_train_loss, 4),
          "| Val Loss:", round(avg_val_loss, 4),
          "| Train F1:", round(train_f1, 4),
          "| Val F1:", round(val_f1, 4))

    if no_improve >= PATIENCE:
        print("Early stopping triggered!")
        break

print()
print("Continue Training Complete!")
print("Best Val F1:", round(best_f1, 4))
print("New best model saved to:", best_path)
wandb.finish()"""

In [ ]:
"""# ============================================================
# EFFICIENTNET-B3 INFERENCE — FULLY SELF CONTAINED
# comment out entire notebook and run only this block
# ============================================================

import os, random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import timm
from tqdm import tqdm

GENRES       = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2IDX    = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE    = {i: g for g, i in GENRE2IDX.items()}
NUM_CLASSES  = 10
SAMPLE_RATE  = 22050
DURATION     = 30
N_MELS       = 128
N_FFT        = 2048
HOP_LENGTH   = 512
TARGET_SHAPE = (128, 512)
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR     = '/kaggle/working'

BASE_DIR   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
TEST_CSV   = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv'
SUB_CSV    = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv'
MODEL_PATH = '/kaggle/working/best_efficientnet_b3-2.pth'

MEAN = -52.4154
STD  =  19.3747

print("Device:", DEVICE)
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


def wav_to_mel(audio):
    mel    = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db


def fix_shape(mel):
    n_mels, time_frames = TARGET_SHAPE
    if mel.shape[1] >= time_frames:
        mel = mel[:, :time_frames]
    else:
        mel = np.pad(mel, ((0, 0), (0, time_frames - mel.shape[1])), mode='constant')
    return mel


def multicrop_inference(model, file_path):
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
    target_len = SAMPLE_RATE * DURATION

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    total    = len(audio)
    crop_len = int(total * 0.8)
    starts   = [0, (total - crop_len) // 2, total - crop_len]

    vote_scores = np.zeros(NUM_CLASSES)
    for start in starts:
        crop = audio[start : start + crop_len]
        mel  = wav_to_mel(crop)
        mel  = fix_shape(mel)
        mel  = torch.tensor(mel, dtype=torch.float32)
        mel  = (mel - MEAN) / (STD + 1e-6)
        mel  = mel.unsqueeze(0).repeat(3, 1, 1)
        mel  = mel.unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            vote_scores += torch.softmax(model(mel), dim=1).cpu().numpy()[0]
    return vote_scores


eff_model = timm.create_model(
    'efficientnet_b3',
    pretrained  = False,
    num_classes = NUM_CLASSES,
    in_chans    = 3
)

eff_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
eff_model = eff_model.to(DEVICE)
eff_model.eval()
print("Model loaded from:", MODEL_PATH)

test_df = pd.read_csv(TEST_CSV)
print("Total test files:", len(test_df))

predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    file_path = os.path.join(BASE_DIR, row['filename'])

    if not os.path.exists(file_path):
        print("Missing:", file_path)
        predictions.append({'id': row['id'], 'genre': 'pop'})
        continue

    try:
        scores    = multicrop_inference(eff_model, file_path)
        predicted = IDX2GENRE[np.argmax(scores)]
        predictions.append({'id': row['id'], 'genre': predicted})
    except Exception as e:
        print("Error on", row['filename'], ":", e)
        predictions.append({'id': row['id'], 'genre': 'pop'})

submission_df = pd.DataFrame(predictions).sort_values('id').reset_index(drop=True)
submission_df.to_csv(os.path.join(SAVE_DIR, 'submission_efficientnet.csv'), index=False)

print()
print("Genre distribution:")
print(submission_df['genre'].value_counts())
print()
print("Saved submission_efficientnet.csv!")
print("DONE!")"""

In [ ]:
#for this code block ,i have downloaded my best_efficient_b3.pyh , give me 1 independent code block and should i upload that downloaded file anywhere ? , so that if i comment out whole notebook , it runs without any issue 

In [ ]:
"""# ============================================================
# ENSEMBLE INFERENCE — CNN + RESNET50 + EFFICIENTNET-B3
# FULLY SELF CONTAINED — comment out entire notebook
# ============================================================

import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torchvision import models
import timm
from tqdm import tqdm

GENRES       = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2IDX    = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE    = {i: g for g, i in GENRE2IDX.items()}
NUM_CLASSES  = 10
SAMPLE_RATE  = 22050
DURATION     = 30
N_MELS       = 128
N_FFT        = 2048
HOP_LENGTH   = 512
TARGET_SHAPE = (128, 512)
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR     = '/kaggle/working'

BASE_DIR  = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
TEST_CSV  = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv'

CNN_PATH  = '/kaggle/input/datasets/shalini784683raaih/messhed-up/best_cnn (1).pth'
RN50_PATH = '/kaggle/input/datasets/shalini784683raaih/resnet4/best_resnet50 (4).pth'
EFF_PATH  = '/kaggle/input/datasets/shalini784683raaih/messhed-up/best_efficientnet_b3.pth'

MEAN = -52.4154
STD  =  19.3747

print("Device:", DEVICE)
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


# ── Audio Functions ───────────────────────────────────────────
def wav_to_mel(audio):
    mel    = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=1.0)
    return mel_db


def fix_shape(mel):
    n_mels, time_frames = TARGET_SHAPE
    if mel.shape[1] >= time_frames:
        mel = mel[:, :time_frames]
    else:
        mel = np.pad(mel, ((0, 0), (0, time_frames - mel.shape[1])), mode='constant')
    return mel


def get_crops(file_path):
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
    target_len = SAMPLE_RATE * DURATION

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    total    = len(audio)
    crop_len = int(total * 0.8)
    starts   = [0, (total - crop_len) // 2, total - crop_len]

    crops = []
    for start in starts:
        crop = audio[start : start + crop_len]
        mel  = wav_to_mel(crop)
        mel  = fix_shape(mel)
        mel  = torch.tensor(mel, dtype=torch.float32)
        mel  = (mel - MEAN) / (STD + 1e-6)
        crops.append(mel)

    return crops


def run_inference(model, crops, three_channel=True):
    vote_scores = np.zeros(NUM_CLASSES)
    for mel in crops:
        if three_channel:
            inp = mel.unsqueeze(0).repeat(3, 1, 1).unsqueeze(0).to(DEVICE)
        else:
            inp = mel.unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            vote_scores += torch.softmax(model(inp), dim=1).cpu().numpy()[0]
    return vote_scores


# ── CNN Model ────────────────────────────────────────────────
class GenreCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(GenreCNN, self).__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        x = self.classifier(x)
        return x


# ── Load All Models ───────────────────────────────────────────
print()
print("Loading models...")

cnn_model = GenreCNN(num_classes=NUM_CLASSES)
cnn_model.load_state_dict(torch.load(CNN_PATH, map_location=DEVICE))
cnn_model = cnn_model.to(DEVICE)
cnn_model.eval()
print("CNN loaded!")

resnet_model       = models.resnet50(weights=None)
resnet_model.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet_model.fc    = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(2048, 512),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(512, NUM_CLASSES)
)
resnet_model.load_state_dict(torch.load(RN50_PATH, map_location=DEVICE))
resnet_model = resnet_model.to(DEVICE)
resnet_model.eval()
print("ResNet50 loaded!")

eff_model = timm.create_model(
    'efficientnet_b3',
    pretrained  = False,
    num_classes = NUM_CLASSES,
    in_chans    = 3
)
eff_model.load_state_dict(torch.load(EFF_PATH, map_location=DEVICE))
eff_model = eff_model.to(DEVICE)
eff_model.eval()
print("EfficientNet-B3 loaded!")

# ── Ensemble Inference ────────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
print()
print("Total test files:", len(test_df))
print("Running ensemble inference...")
print()

predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Ensemble"):
    file_path = os.path.join(BASE_DIR, row['filename'])

    if not os.path.exists(file_path):
        print("Missing:", file_path)
        predictions.append({'id': row['id'], 'genre': 'pop'})
        continue

    try:
        crops = get_crops(file_path)

        cnn_scores    = run_inference(cnn_model,    crops, three_channel=False)
        resnet_scores = run_inference(resnet_model, crops, three_channel=True)
        eff_scores    = run_inference(eff_model,    crops, three_channel=True)

        # Soft voting — sum all scores
        # ResNet and EfficientNet weighted higher as they are stronger models
        ensemble  = cnn_scores + resnet_scores * 2 + eff_scores * 2

        predicted = IDX2GENRE[np.argmax(ensemble)]
        predictions.append({'id': row['id'], 'genre': predicted})

    except Exception as e:
        print("Error on", row['filename'], ":", e)
        predictions.append({'id': row['id'], 'genre': 'pop'})

submission_df = pd.DataFrame(predictions).sort_values('id').reset_index(drop=True)
submission_df.to_csv(os.path.join(SAVE_DIR, 'submission_ensemble.csv'), index=False)

print()
print("Genre distribution:")
print(submission_df['genre'].value_counts())
print()
print("Saved submission.csv!")
print("DONE! Submit this for best score!")"""

Device: cuda
GPU   : Tesla T4

Loading models...
CNN loaded!
ResNet50 loaded!
EfficientNet-B3 loaded!

Total test files: 3020
Running ensemble inference...

Ensemble: 100%|██████████| 3020/3020 [12:14<00:00,  4.11it/s]

Genre distribution:
genre
rock         442
hiphop       337
pop          318
reggae       308
disco        307
classical    290
metal        290
jazz         281
blues        227
country      220
Name: count, dtype: int64

Saved submission.csv!
DONE! Submit this for best score!

In [ ]:
"""# ============================================================
# ENSEMBLE INFERENCE — CNN + EFFICIENTNET-B3
# FULLY SELF CONTAINED — comment out entire notebook
# ============================================================

import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import timm
from tqdm import tqdm

GENRES       = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
GENRE2IDX    = {g: i for i, g in enumerate(GENRES)}
IDX2GENRE    = {i: g for g, i in GENRE2IDX.items()}
NUM_CLASSES  = 10
SAMPLE_RATE  = 22050
DURATION     = 30
N_MELS       = 128
N_FFT        = 2048
HOP_LENGTH   = 512
TARGET_SHAPE = (128, 512)
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR     = '/kaggle/working'

BASE_DIR  = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
TEST_CSV  = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv'

CNN_PATH  = '/kaggle/input/datasets/shalini784683raaih/messhed-up/best_cnn (1).pth'
EFF_PATH  = '/kaggle/input/datasets/shalini784683raaih/messhed-up/best_efficientnet_b3.pth'

MEAN = -52.4154
STD  =  19.3747

print("Device:", DEVICE)
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


def wav_to_mel(audio):
    mel    = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db


def fix_shape(mel):
    n_mels, time_frames = TARGET_SHAPE
    if mel.shape[1] >= time_frames:
        mel = mel[:, :time_frames]
    else:
        mel = np.pad(mel, ((0, 0), (0, time_frames - mel.shape[1])), mode='constant')
    return mel


def get_crops(file_path):
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
    target_len = SAMPLE_RATE * DURATION

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    total    = len(audio)
    crop_len = int(total * 0.8)
    starts   = [0, (total - crop_len) // 2, total - crop_len]

    crops = []
    for start in starts:
        crop = audio[start : start + crop_len]
        mel  = wav_to_mel(crop)
        mel  = fix_shape(mel)
        mel  = torch.tensor(mel, dtype=torch.float32)
        mel  = (mel - MEAN) / (STD + 1e-6)
        crops.append(mel)

    return crops


def run_inference(model, crops, three_channel=True):
    vote_scores = np.zeros(NUM_CLASSES)
    for mel in crops:
        if three_channel:
            inp = mel.unsqueeze(0).repeat(3, 1, 1).unsqueeze(0).to(DEVICE)
        else:
            inp = mel.unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            vote_scores += torch.softmax(model(inp), dim=1).cpu().numpy()[0]
    return vote_scores


# ── CNN Model ────────────────────────────────────────────────
class GenreCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(GenreCNN, self).__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.conv_block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        x = self.classifier(x)
        return x


# ── Load Models ───────────────────────────────────────────────
print()
print("Loading models...")

cnn_model = GenreCNN(num_classes=NUM_CLASSES)
cnn_model.load_state_dict(torch.load(CNN_PATH, map_location=DEVICE))
cnn_model = cnn_model.to(DEVICE)
cnn_model.eval()
print("CNN loaded!")

eff_model = timm.create_model(
    'efficientnet_b3',
    pretrained  = False,
    num_classes = NUM_CLASSES,
    in_chans    = 3
)
eff_model.load_state_dict(torch.load(EFF_PATH, map_location=DEVICE))
eff_model = eff_model.to(DEVICE)
eff_model.eval()
print("EfficientNet-B3 loaded!")

# ── Ensemble Inference ────────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
print()
print("Total test files:", len(test_df))
print("Running CNN + EfficientNet ensemble...")
print()

predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Ensemble"):
    file_path = os.path.join(BASE_DIR, row['filename'])

    if not os.path.exists(file_path):
        print("Missing:", file_path)
        predictions.append({'id': row['id'], 'genre': 'pop'})
        continue

    try:
        crops = get_crops(file_path)

        cnn_scores = run_inference(cnn_model, crops, three_channel=False)
        eff_scores = run_inference(eff_model, crops, three_channel=True)

        # EfficientNet weighted higher as it is the stronger model
        ensemble  = cnn_scores + eff_scores * 2

        predicted = IDX2GENRE[np.argmax(ensemble)]
        predictions.append({'id': row['id'], 'genre': predicted})

    except Exception as e:
        print("Error on", row['filename'], ":", e)
        predictions.append({'id': row['id'], 'genre': 'pop'})

submission_df = pd.DataFrame(predictions).sort_values('id').reset_index(drop=True)
submission_df.to_csv(os.path.join(SAVE_DIR, 'submission.csv'), index=False)

print()
print("Genre distribution:")
print(submission_df['genre'].value_counts())
print()
print("Saved submission_ensemble.csv!")
print("DONE! Submit this!")"""